# 02 — NOAA Weather Data
Pulls monthly TAVG and PRCP for 5 states, 2005–2024, from NOAA CDO API.
Output: `data/raw/noaa_weather.csv` and `data/processed/weather_features.csv`

In [1]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.getenv("NOAA_API_KEY")
assert API_KEY, "NOAA_API_KEY not found in .env"
print("API key loaded.")

/Users/devinlucas/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


API key loaded.


In [2]:
STATES = {
    "IOWA":      "FIPS:19",
    "COLORADO":  "FIPS:08",
    "WISCONSIN": "FIPS:55",
    "MISSOURI":  "FIPS:29",
    "NEBRASKA":  "FIPS:31"
}

# Growing season months: May through October
MONTHS = ["05", "06", "07", "08", "09", "10"]

records = []

for state_name, state_fips in STATES.items():
    for year in range(2005, 2026):
        for month in MONTHS:
            url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
            params = {
                "datasetid": "GSOM",
                "datatypeid": "TAVG,PRCP",
                "locationid": state_fips,
                "startdate": f"{year}-{month}-01",
                "enddate": f"{year}-{month}-01",
                "units": "standard",
                "limit": 100
            }
            headers = {"token": API_KEY}
            try:
                r = requests.get(url, params=params, headers=headers, timeout=10)
                if r.status_code == 200 and "results" in r.json():
                    for item in r.json()["results"]:
                        item["state"] = state_name
                        item["year"] = year
                        item["month"] = int(month)
                        records.append(item)
                elif r.status_code != 200:
                    print(f"  HTTP {r.status_code}: {state_name} {year}-{month}")
            except requests.exceptions.Timeout:
                print(f"  TIMEOUT: {state_name} {year}-{month}, skipping")
            except Exception as e:
                print(f"  ERROR: {state_name} {year}-{month}: {e}")
            time.sleep(0.2)
        print(f"Done: {state_name} {year}")

weather_raw = pd.DataFrame(records)
weather_raw.to_csv("../data/raw/noaa_weather.csv", index=False)
print(f"Saved {len(weather_raw)} raw rows")
print(weather_raw.head())

Done: IOWA 2005
Done: IOWA 2006
Done: IOWA 2007
Done: IOWA 2008
Done: IOWA 2009
Done: IOWA 2010
Done: IOWA 2011
Done: IOWA 2012
Done: IOWA 2013
Done: IOWA 2014
Done: IOWA 2015
Done: IOWA 2016
Done: IOWA 2017
Done: IOWA 2018
Done: IOWA 2019
Done: IOWA 2020
Done: IOWA 2021
Done: IOWA 2022
Done: IOWA 2023
Done: IOWA 2024
Done: IOWA 2025
Done: COLORADO 2005
Done: COLORADO 2006
Done: COLORADO 2007
Done: COLORADO 2008
Done: COLORADO 2009
Done: COLORADO 2010
Done: COLORADO 2011
Done: COLORADO 2012
Done: COLORADO 2013
Done: COLORADO 2014
Done: COLORADO 2015
Done: COLORADO 2016
Done: COLORADO 2017
Done: COLORADO 2018
Done: COLORADO 2019
Done: COLORADO 2020
Done: COLORADO 2021
Done: COLORADO 2022
Done: COLORADO 2023
Done: COLORADO 2024
Done: COLORADO 2025
Done: WISCONSIN 2005
Done: WISCONSIN 2006
Done: WISCONSIN 2007
Done: WISCONSIN 2008
Done: WISCONSIN 2009
Done: WISCONSIN 2010
Done: WISCONSIN 2011
Done: WISCONSIN 2012
Done: WISCONSIN 2013
Done: WISCONSIN 2014
Done: WISCONSIN 2015
Done: WISCONS

In [3]:
# Pivot to wide format: one row per state per year
# Columns: tavg_may, prcp_may, tavg_jun, prcp_jun, ...

MONTH_NAMES = {
    5: "may", 6: "jun", 7: "jul",
    8: "aug", 9: "sep", 10: "oct"
}

weather_raw = pd.read_csv("../data/raw/noaa_weather.csv")

# Pivot datatype (TAVG/PRCP) into columns
weather_pivot = weather_raw.pivot_table(
    index=['year', 'state', 'month'],
    columns='datatype',
    values='value',
    aggfunc='mean'
).reset_index()

weather_pivot.columns.name = None
weather_pivot['month_name'] = weather_pivot['month'].map(MONTH_NAMES)

# Build wide format
rows = []
for (year, state), grp in weather_pivot.groupby(['year', 'state']):
    row = {'year': year, 'state': state}
    for _, r in grp.iterrows():
        mn = r['month_name']
        if 'TAVG' in r: row[f'tavg_{mn}'] = r['TAVG']
        if 'PRCP' in r: row[f'prcp_{mn}'] = r['PRCP']
    rows.append(row)

weather_features = pd.DataFrame(rows)
weather_features.to_csv("../data/processed/weather_features.csv", index=False)
print(f"Saved {len(weather_features)} rows")
print(weather_features.head())

Saved 105 rows
   year      state   tavg_may  prcp_may   tavg_jun  prcp_jun   tavg_jul  \
0  2005   COLORADO        NaN  1.409300        NaN  2.783200        NaN   
1  2005       IOWA  57.970000  3.820167  72.914286  5.026207  74.956410   
2  2005   MISSOURI  63.171053  1.987258  75.189474  3.811290  78.394872   
3  2005   NEBRASKA        NaN  3.225500        NaN  4.926700        NaN   
4  2005  WISCONSIN  53.517500  2.656333  70.147619  3.798793  71.244186   

   prcp_jul   tavg_aug  prcp_aug   tavg_sep  prcp_sep   tavg_oct  prcp_oct  
0  0.686600        NaN  2.303700        NaN  0.533600        NaN  2.386200  
1  3.168852  72.040000  3.574667  67.756098  3.235085  52.717073  0.901525  
2  3.028197  78.263158  5.532742  72.289744  3.990656  57.128205  1.542787  
3  1.693300        NaN  3.161000        NaN  0.498700        NaN  1.846700  
4  3.360702  69.082927  2.910339  64.037209  3.975439  50.361905  2.723793  
